# Travel planner agent (smolagents + Gemini)

**How to run**
1. In Colab, open the key icon (Secrets), add `GEMINI_API_KEY`, and turn on notebook access.
2. Run cell 1 (install), then **Runtime -> Restart session**. Do not run the install cell again afterwards.
3. Run the remaining cells from top to bottom.

Design: the agents only gather data (coordinates, nearby baseball stadium, flight time) and return a dataframe.
The map is built and exported in normal Python afterwards, so it does not depend on the agent sandbox or its 30 second limit.

In [ ]:
# Cell 1: install, then restart the session (Runtime -> Restart session)
!pip install -q -U 'smolagents[litellm]' "plotly>=6.1.1,<7" "kaleido>=1.0" geopandas shapely pandas markdownify requests ddgs pillow
!plotly_get_chrome -y

Restart the session now, then continue from the next cell.

In [ ]:
# Cell 2: imports, API key, model settings
import os, math, time, glob, shutil, subprocess
import pandas as pd
import plotly
import plotly.express as px
from PIL import Image
from google.colab import userdata
from smolagents import CodeAgent, LiteLLMModel, DuckDuckGoSearchTool, VisitWebpageTool, tool

print("plotly", plotly.__version__)
assert int(plotly.__version__.split(".")[0]) >= 6, (
    "Plotly 5.x is still loaded. Use Runtime -> Restart session and do NOT run the install cell again."
)

# Key comes from Colab Secrets, never paste it into a cell
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# If you get empty replies or 503s, try "gemini/gemini-3.7-flash" or "gemini/gemini-3.8-flash"
MODEL_ID = "gemini/gemini-2.5-flash"


class RobustLiteLLM(LiteLLMModel):
    """LiteLLMModel that retries when Gemini returns an empty message."""

    def generate(self, *args, **kwargs):
        msg = None
        for _ in range(5):
            msg = super().generate(*args, **kwargs)
            if msg.content:
                return msg
            time.sleep(3)
        return msg


def make_model():
    return RobustLiteLLM(MODEL_ID, max_tokens=8096, num_retries=5)

In [ ]:
# Cell 3: custom flight-time tool
@tool
def calculate_cargo_travel_time(
    origin_coords: tuple[float, float],
    destination_coords: tuple[float, float],
    cruising_speed_kmh: float = 750.0,
) -> float:
    """
    Calculate the travel time for a plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Cruising speed in km/h (default 750 for cargo planes, use about 850 for passenger planes)

    Returns:
        float: The estimated travel time in hours, including a 10% route buffer and 1 hour of takeoff and landing overhead
    """
    lat1, lon1 = map(math.radians, origin_coords)
    lat2, lon2 = map(math.radians, destination_coords)

    a = (
        math.sin((lat2 - lat1) / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    )
    distance_km = 6371.0 * 2 * math.asin(math.sqrt(a))

    return round((distance_km * 1.1) / cruising_speed_kmh + 1.0, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))  # expect 22.82

In [ ]:
# Cell 4: web agent + manager agent
web_agent = CodeAgent(
    model=make_model(),
    tools=[DuckDuckGoSearchTool(), VisitWebpageTool()],
    name="web_agent",
    description="Browses the web to find facts such as coordinates and nearby places.",
    verbosity_level=0,
    max_steps=8,
)

manager_kwargs = dict(
    model=make_model(),
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=["pandas", "numpy", "math", "datetime", "json"],
    planning_interval=None,
    verbosity_level=1,
    max_steps=12,
)

# The default 30 s code limit is too short while the web agent is working
try:
    manager_agent = CodeAgent(**manager_kwargs, executor_kwargs={"timeout_seconds": 300})
except TypeError:
    manager_agent = CodeAgent(**manager_kwargs)
    print("Note: this smolagents version has no timeout option; keep the web task small.")

In [ ]:
# Cell 5: run the agents (returns a dataframe)
TASK = """
Stadiums: MetLife Stadium (NJ), Lincoln Financial Field (PA), Gillette Stadium (MA).

Do these steps in order:
1. Call web_agent ONCE with a single short task: ask for the latitude and longitude (decimal degrees,
   negative longitude for west) of all three stadiums, plus one nearby Major League Baseball stadium
   in the same region for each. Print its reply and read it yourself.
2. For each stadium, call calculate_cargo_travel_time with origin_coords=(40.7128, -74.0060) (New York City),
   the stadium's coordinates as destination_coords, and cruising_speed_kmh=850 (passenger plane).
3. Build a pandas dataframe with exactly these columns:
   Stadium Name, Latitude, Longitude, Travel Time to NYC, Nearby Baseball Stadium.
   Latitude and Longitude must be floats. Travel Time to NYC must be a string formatted HH:MM:SS
   (zero-padded hours, minutes, seconds converted from the decimal hours).
4. Call final_answer(df) with the dataframe. Do not plot anything.

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
"""

result = manager_agent.run(TASK)

try:
    df = result if isinstance(result, pd.DataFrame) else pd.DataFrame(result)
except Exception:
    print("The agent did not return a table. Raw result:")
    print(result)
    raise

df

In [ ]:
# Cell 6: build the map (plain Python, outside the agent sandbox)
def to_hours(hhmmss):
    h, m, s = (int(x) for x in str(hhmmss).split(":"))
    return h + m / 60 + s / 3600


df[["Latitude", "Longitude"]] = df[["Latitude", "Longitude"]].astype(float)
df["Travel Hours"] = df["Travel Time to NYC"].apply(to_hours)

fig = px.scatter_map(
    df,
    lat="Latitude",
    lon="Longitude",
    text="Stadium Name",
    hover_name="Stadium Name",
    hover_data=["Travel Time to NYC", "Nearby Baseball Stadium"],
    color="Travel Hours",
    color_continuous_scale="Magma",
    size=[15] * len(df),
    size_max=15,
    zoom=5,
    center={"lat": df["Latitude"].mean(), "lon": df["Longitude"].mean()},
    title="Stadiums coloured by flight time from NYC",
)
fig.show()

In [ ]:
# Cell 7: export to PNG, with fallbacks if Kaleido fails
def save_map_png(fig, path="saved_map.png", width=1000, height=650):
    if os.path.exists(path):
        os.remove(path)

    # 1) Kaleido (needs plotly >= 6.1 and Chrome, installed in cell 1)
    try:
        fig.write_image(path, width=width, height=height)
        return "kaleido"
    except Exception as e:
        print("Kaleido export failed:", repr(e)[:200])

    # 2) Headless Chrome screenshot of an HTML copy of the map
    chrome = (
        shutil.which("chromium")
        or shutil.which("chromium-browser")
        or shutil.which("google-chrome")
        or next(iter(glob.glob("/root/.local/share/choreographer/deps/*/chrome")), None)
    )
    if chrome:
        fig.write_html("temp_map.html", include_plotlyjs="cdn")
        subprocess.run(
            [
                chrome, "--headless=new", "--no-sandbox", "--disable-gpu", "--hide-scrollbars",
                f"--window-size={width},{height}", "--virtual-time-budget=15000",
                f"--screenshot={os.path.abspath(path)}",
                "file://" + os.path.abspath("temp_map.html"),
            ],
            capture_output=True, timeout=120,
        )
        if os.path.exists(path):
            return "chrome"

    # 3) Last resort: interactive HTML
    fig.write_html("saved_map.html")
    return "html"


method = save_map_png(fig)
print("Saved using:", method)
if os.path.exists("saved_map.png"):
    display(Image.open("saved_map.png"))

In [ ]:
# Cell 8 (optional): ask Gemini to review the map image
import base64
from io import BytesIO

def check_map_with_gemini(df, path="saved_map.png"):
    if not os.path.exists(path):
        print("No PNG to check (export used HTML fallback).")
        return
    buf = BytesIO()
    Image.open(path).convert("RGB").save(buf, format="PNG")
    data_url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
    prompt = (
        "Here is a dataframe of stadiums and flight times from New York City:\n"
        + df.drop(columns=["Travel Hours"]).to_string(index=False)
        + "\n\nHere is the map that was made. Does it show all three stadiums as scatter points "
        "coloured by travel time? List reasons, then finish with PASS or FAIL."
    )
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": data_url}},
        ],
    }]
    print(make_model()(messages).content)


check_map_with_gemini(df)